In [1]:
# Model Training for Stock Price Prediction
# Training 4 models: Logistic Regression, Random Forest, XGBoost, LSTM
# Using TimeSeriesSplit for proper time series evaluation

import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ML Libraries
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# XGBoost
from xgboost import XGBClassifier, XGBRegressor

# PyTorch for Neural Networks
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
plt.style.use('default')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Analyze learning curves and hyperparameter effects

def plot_learning_curves_for_lstm(stock_name, X, y):
    """Plot learning curves showing batch_size and learning_rate effects"""
    print(f"\nAnalyzing learning curves for {stock_name} LSTM...")

    # Prepare data
    X_seq, y_seq = prepare_lstm_data(X, y, time_steps=10)
    if len(X_seq) == 0:
        print("Not enough data for LSTM analysis")
        return

    # Convert to PyTorch tensors
    X_seq = torch.tensor(X_seq, dtype=torch.float32)
    y_seq = torch.tensor(y_seq, dtype=torch.float32).unsqueeze(1)

    # Test different batch sizes and learning rates
    batch_sizes = [16, 32, 64]
    learning_rates = [0.001, 0.01, 0.1]

    results = {}

    for batch_size in batch_sizes:
        for lr in learning_rates:
            print(f"Testing batch_size={batch_size}, learning_rate={lr}")

            model, optimizer, criterion = create_lstm_model(
                input_size=X_seq.shape[2],
                units=64,
                dropout=0.2,
                learning_rate=lr
            )

            # Use only the last split for quick analysis
            train_idx, val_idx = list(tscv.split(X_seq))[-1]
            X_train, X_val = X_seq[train_idx], X_seq[val_idx]
            y_train, y_val = y_seq[train_idx], y_seq[val_idx]

            train_dataset = TensorDataset(X_train, y_train)
            val_dataset = TensorDataset(X_val, y_val)
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

            # Track losses and accuracies
            train_losses = []
            val_losses = []
            train_accuracies = []
            val_accuracies = []

            model.train()
            for epoch in range(50):  # Fixed epochs for analysis
                epoch_train_loss = 0
                epoch_train_correct = 0
                epoch_train_total = 0

                for inputs, targets in train_loader:
                    optimizer.zero_grad()
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()

                    epoch_train_loss += loss.item()
                    preds = (outputs > 0.5).float()
                    epoch_train_correct += (preds == targets).sum().item()
                    epoch_train_total += targets.size(0)

                train_losses.append(epoch_train_loss / len(train_loader))
                train_accuracies.append(epoch_train_correct / epoch_train_total)

                # Validation
                model.eval()
                epoch_val_loss = 0
                epoch_val_correct = 0
                epoch_val_total = 0

                with torch.no_grad():
                    for inputs, targets in val_loader:
                        outputs = model(inputs)
                        loss = criterion(outputs, targets)

                        epoch_val_loss += loss.item()
                        preds = (outputs > 0.5).float()
                        epoch_val_correct += (preds == targets).sum().item()
                        epoch_val_total += targets.size(0)

                val_losses.append(epoch_val_loss / len(val_loader))
                val_accuracies.append(epoch_val_correct / epoch_val_total)

                model.train()

            key = f"bs_{batch_size}_lr_{lr}"
            results[key] = {
                'batch_size': batch_size,
                'learning_rate': lr,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'train_accuracies': train_accuracies,
                'val_accuracies': val_accuracies,
                'epochs': len(train_losses)
            }

    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'LSTM Learning Curves Analysis - {stock_name}', fontsize=16)

    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive']

    for i, (key, result) in enumerate(results.items()):
        color = colors[i % len(colors)]
        train_losses = result['train_losses']
        val_losses = result['val_losses']
        train_accuracies = result['train_accuracies']
        val_accuracies = result['val_accuracies']

        # Loss curves
        axes[0, 0].plot(train_losses, label=f'{key} (train)', color=color, linestyle='-')
        axes[0, 0].plot(val_losses, label=f'{key} (val)', color=color, linestyle='--')

        # Accuracy curves
        axes[0, 1].plot(train_accuracies, label=f'{key} (train)', color=color, linestyle='-')
        axes[0, 1].plot(val_accuracies, label=f'{key} (val)', color=color, linestyle='--')

    axes[0, 0].set_title('Training vs Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].set_title('Training vs Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[0, 1].grid(True, alpha=0.3)

    # Analyze convergence by batch size
    batch_size_analysis = {}
    for bs in batch_sizes:
        bs_results = [r for r in results.values() if r['batch_size'] == bs]
        avg_final_loss = np.mean([r['val_losses'][-1] for r in bs_results])
        avg_epochs = np.mean([r['epochs'] for r in bs_results])
        batch_size_analysis[bs] = {'avg_final_loss': avg_final_loss, 'avg_epochs': avg_epochs}

    # Plot batch size effects
    bs_sizes = list(batch_size_analysis.keys())
    bs_losses = [batch_size_analysis[bs]['avg_final_loss'] for bs in bs_sizes]
    bs_epochs = [batch_size_analysis[bs]['avg_epochs'] for bs in bs_sizes]

    axes[1, 0].bar(bs_sizes, bs_losses, color='skyblue', alpha=0.7)
    axes[1, 0].set_title('Average Final Validation Loss by Batch Size')
    axes[1, 0].set_xlabel('Batch Size')
    axes[1, 0].set_ylabel('Average Final Validation Loss')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].bar(bs_sizes, bs_epochs, color='lightcoral', alpha=0.7)
    axes[1, 1].set_title('Average Training Epochs by Batch Size')
    axes[1, 1].set_xlabel('Batch Size')
    axes[1, 1].set_ylabel('Average Epochs to Converge')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'../models/{stock_name}_lstm_learning_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print analysis summary
    print(f"\nBatch Size Analysis for {stock_name}:")
    for bs, metrics in batch_size_analysis.items():
        print(f"  Batch Size {bs}: Avg Loss = {metrics['avg_final_loss']:.4f}, Avg Epochs = {metrics['avg_epochs']:.1f}")

# Run learning curve analysis for trained stocks
for stock_name in stocks_to_train:
    if stock_name in all_results and 'LSTM' in all_results[stock_name]:
        df = stock_data[stock_name]
        X = df[feature_cols].values
        y = df['target_classification'].values
        plot_learning_curves_for_lstm(stock_name, X, y)

print("\nLearning curve analysis completed")

# 5. Learning Curves and Hyperparameter Analysis

In [ ]:
# Train models for all stocks
# Note: This may take a long time due to hyperparameter tuning

all_results = {}

# Choose which stocks to train (can be modified)
stocks_to_train = ['FPT']  # Start with one stock for testing

for stock_name in stocks_to_train:
    print(f"\n{'='*80}")
    print(f"STARTING TRAINING FOR {stock_name}")
    print(f"{'='*80}")

    df = stock_data[stock_name]
    results = train_and_evaluate_stock_models(stock_name, df, feature_cols)

    all_results[stock_name] = results

    print(f"\n{'='*80}")
    print(f"COMPLETED TRAINING FOR {stock_name}")
    print(f"{'='*80}")

print(f"\n{'='*100}")
print("MODEL TRAINING COMPLETED FOR ALL SELECTED STOCKS")
print(f"{'='*100}")

# Save results
import pickle
results_file = r'../models/training_results.pkl'
os.makedirs(os.path.dirname(results_file), exist_ok=True)
with open(results_file, 'wb') as f:
    pickle.dump(all_results, f)

print(f"\nTraining results saved to: {results_file}")

# Summary of results
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")

for stock_name, stock_results in all_results.items():
    print(f"\n{stock_name}:")
    for model_name, result in stock_results.items():
        score = result['best_score']
        print("4f")


STARTING TRAINING FOR FPT

Training models for FPT

--- Training LogisticRegression ---
Best parameters: {'solver': 'lbfgs', 'max_iter': 5000, 'C': 10}
.4f

--- Training RandomForest ---
Best parameters: {'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 10}
.4f

--- Training XGBoost ---
Best parameters: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
.4f

--- Training LSTM ---


# 4. Train Models for All Stocks

In [4]:
# Training and evaluation functions

def evaluate_classification(y_true, y_pred):
    """Evaluate classification model performance"""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1': f1_score(y_true, y_pred, average='weighted')
    }

def evaluate_regression(y_true, y_pred):
    """Evaluate regression model performance"""
    mse = mean_squared_error(y_true, y_pred)
    return {
        'mse': mse,
        'rmse': np.sqrt(mse),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred)
    }

def train_sklearn_model(X, y, model_config, task='classification'):
    """Train sklearn model with TimeSeriesSplit and hyperparameter tuning"""
    model_class = model_config['model_class']
    param_grid = model_config['param_grid']

    # Use RandomizedSearchCV for efficiency
    search_cv = RandomizedSearchCV(
        model_class(random_state=42),
        param_grid,
        cv=tscv,
        scoring='accuracy' if task == 'classification' else 'neg_mean_squared_error',
        n_iter=20,  # Limit iterations for speed
        random_state=42,
        n_jobs=-1
    )

    # Fit the model
    search_cv.fit(X, y)

    return search_cv.best_estimator_, search_cv.best_params_, search_cv.best_score_

def train_lstm_model(X, y, param_grid):
    """Train LSTM model with hyperparameter tuning using PyTorch"""
    best_score = -np.inf
    best_model = None
    best_params = None

    # Prepare data for LSTM (create sequences)
    X_seq, y_seq = prepare_lstm_data(X, y, time_steps=10)

    if len(X_seq) == 0:
        return None, None, None

    # Convert to PyTorch tensors
    X_seq = torch.tensor(X_seq, dtype=torch.float32)
    y_seq = torch.tensor(y_seq, dtype=torch.float32).unsqueeze(1)

    # Grid search over hyperparameters
    for batch_size in param_grid['batch_size']:
        for learning_rate in param_grid['learning_rate']:
            for epochs in param_grid['epochs']:
                for units in param_grid['units']:
                    for dropout in param_grid['dropout']:
                        try:
                            # Create model
                            model, optimizer, criterion = create_lstm_model(
                                input_size=X_seq.shape[2],
                                units=units,
                                dropout=dropout,
                                learning_rate=learning_rate
                            )

                            # Train model with TimeSeriesSplit
                            scores = []
                            for train_idx, val_idx in tscv.split(X_seq):
                                X_train, X_val = X_seq[train_idx], X_seq[val_idx]
                                y_train, y_val = y_seq[train_idx], y_seq[val_idx]

                                # Create DataLoaders
                                train_dataset = TensorDataset(X_train, y_train)
                                val_dataset = TensorDataset(X_val, y_val)
                                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
                                val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

                                # Training loop
                                model.train()
                                for epoch in range(epochs):
                                    for inputs, targets in train_loader:
                                        optimizer.zero_grad()
                                        outputs = model(inputs)
                                        loss = criterion(outputs, targets)
                                        loss.backward()
                                        optimizer.step()

                                # Validation
                                model.eval()
                                val_preds = []
                                val_targets = []
                                with torch.no_grad():
                                    for inputs, targets in val_loader:
                                        outputs = model(inputs)
                                        val_preds.extend(outputs.cpu().numpy())
                                        val_targets.extend(targets.cpu().numpy())

                                val_preds = np.array(val_preds)
                                val_targets = np.array(val_targets)
                                val_accuracy = accuracy_score((val_targets > 0.5).astype(int), (val_preds > 0.5).astype(int))
                                scores.append(val_accuracy)

                            avg_score = np.mean(scores)

                            if avg_score > best_score:
                                best_score = avg_score
                                best_model = model
                                best_params = {
                                    'batch_size': batch_size,
                                    'learning_rate': learning_rate,
                                    'epochs': epochs,
                                    'units': units,
                                    'dropout': dropout
                                }

                        except Exception as e:
                            print(f"Error training LSTM with params {locals()}: {e}")
                            continue

    return best_model, best_params, best_score

def train_and_evaluate_stock_models(stock_name, df, feature_cols):
    """Train and evaluate all models for a single stock"""
    print(f"\n{'='*60}")
    print(f"Training models for {stock_name}")
    print(f"{'='*60}")

    # Prepare data
    X = df[feature_cols].values
    y_class = df['target_classification'].values
    y_reg = df['target_regression'].values

    results = {}

    # Train each model (skip LSTM for now due to long training time)
    for model_name, config in model_configs.items():
        if model_name == 'LSTM':
            print(f"\n--- Skipping {model_name} (long training time) ---")
            continue
            
        print(f"\n--- Training {model_name} ---")

        try:
            if model_name == 'LSTM':
                # Train LSTM model
                model, best_params, best_score = train_lstm_model(X, y_class, config['param_grid'])

                if model is None:
                    print(f"Failed to train {model_name}")
                    continue

                results[model_name] = {
                    'model': model,
                    'best_params': best_params,
                    'best_score': best_score,
                    'task': 'classification'
                }

            else:
                # Train sklearn models
                model, best_params, best_score = train_sklearn_model(X, y_class, config, config['task'])

                results[model_name] = {
                    'model': model,
                    'best_params': best_params,
                    'best_score': best_score,
                    'task': config['task']
                }

            print(f"Best parameters: {best_params}")
            print(".4f")

        except Exception as e:
            print(f"Error training {model_name}: {e}")
            continue

    return results

print("Training and evaluation functions defined")

Training and evaluation functions defined


# 3. Training and Evaluation Functions

In [2]:
# Model configurations for both classification and regression
model_configs = {
    'LogisticRegression': {
        'model_class': LogisticRegression,
        'task': 'classification',
        'param_grid': {
            'C': [0.01, 0.1, 1, 10, 100],
            'max_iter': [1000, 2000, 5000],
            'solver': ['lbfgs', 'liblinear']
        }
    },
    'RandomForest': {
        'model_class': RandomForestClassifier,
        'task': 'classification',
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'XGBoost': {
        'model_class': XGBClassifier,
        'task': 'classification',
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 6, 9],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
    },
    'LSTM': {
        'task': 'classification',
        'param_grid': {
            'batch_size': [16, 32, 64],
            'learning_rate': [0.001, 0.01, 0.1],
            'epochs': [50, 100],
            'units': [32, 64, 128],
            'dropout': [0.1, 0.2, 0.3]
        }
    }
}

# PyTorch LSTM Model Class
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.dropout(out[:, -1, :])  # Take the last time step
        out = torch.relu(self.fc1(out))
        out = self.sigmoid(self.fc2(out))
        return out

# Function to create LSTM model
def create_lstm_model(input_size, units=64, dropout=0.2, learning_rate=0.001):
    """Create LSTM model for time series classification"""
    model = LSTMModel(input_size=input_size, hidden_size=units, dropout=dropout)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()
    return model, optimizer, criterion

# Function to prepare data for LSTM
def prepare_lstm_data(X, y, time_steps=10):
    """Prepare data for LSTM by creating sequences"""
    X_seq, y_seq = [], []

    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
        y_seq.append(y[i+time_steps])

    return np.array(X_seq), np.array(y_seq)

print("Model configurations defined:")
for model_name, config in model_configs.items():
    task = config['task']
    params = len(config['param_grid'])
    print(f"  • {model_name}: {task} task, {params} hyperparameters to tune")

Model configurations defined:
  • LogisticRegression: classification task, 3 hyperparameters to tune
  • RandomForest: classification task, 4 hyperparameters to tune
  • XGBoost: classification task, 5 hyperparameters to tune
  • LSTM: classification task, 5 hyperparameters to tune


# 2. Model Definitions and Hyperparameter Grids

In [3]:
# Load feature engineered data for all stocks
feature_dir = r'../data/processed/feature_engineered'
stock_names = ['FPT', 'HPG', 'POW', 'VCB', 'VIC']

# Load data for each stock
stock_data = {}
for stock in stock_names:
    file_path = os.path.join(feature_dir, stock, f'{stock}_features.csv')
    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time').reset_index(drop=True)
    stock_data[stock] = df
    print(f"Loaded {stock}: {df.shape}")

print(f"\nLoaded feature engineered data for {len(stock_data)} stocks")

# Setup TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
print(f"\nTimeSeriesSplit configured with {tscv.n_splits} splits")

# Define feature columns (exclude time and target columns)
exclude_cols = ['time', 'target_classification', 'target_regression']
feature_cols = [col for col in stock_data['FPT'].columns if col not in exclude_cols]

print(f"\nFeature columns ({len(feature_cols)}): {feature_cols}")
print(f"Target columns: target_classification (classification), target_regression (regression)")

Loaded FPT: (1030, 24)
Loaded HPG: (1030, 24)
Loaded POW: (1030, 24)
Loaded VCB: (1030, 24)
Loaded VIC: (1030, 24)

Loaded feature engineered data for 5 stocks

TimeSeriesSplit configured with 5 splits

Feature columns (21): ['open', 'close', 'low', 'high', 'volume', 'SMA_10', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'RSI_14', 'MACD_12_26', 'MACD_signal_9', 'MACD_histogram', 'BB_upper_20', 'BB_middle_20', 'BB_lower_20', 'close_lag_1', 'close_lag_3', 'close_lag_7', 'volatility_10']
Target columns: target_classification (classification), target_regression (regression)


# 1. Load Feature Engineered Data